# Collapse Analysis for AstroDINO Embeddings

This notebook checks if the AstroDINO model has collapsed (all embeddings become similar).

## Collapse Indicators:
1. **Pairwise Cosine Similarity**: Should have wide distribution, not peaked at 1.0
2. **Per-dimension Variance**: Many dimensions with variance ≈ 0 indicates collapse
3. **Effective Dimensionality**: Features compressed to low-dimensional subspace
4. **Nearest Neighbor Distance Distribution**: Should have variation, not constant

In [ ]:
import os
import sys
import numpy as np
import torch
import h5py
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from omegaconf import OmegaConf
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.metrics.pairwise import cosine_similarity

# Add project root to path
PROJECT_ROOT = "/u/yacheng/projects/ssl_outthere"
sys.path.insert(0, PROJECT_ROOT)

from dinov2.eval.setup import build_model_for_eval
from encoder_image.astrodino.train.data.augmentations import ToRGB

In [ ]:
# Configuration for both models
MODELS = {
    'f115w': {
        'config': '/u/yacheng/projects/ssl_outthere/encoder_image/astrodino/model/astrodino_f115w_vitl/config.yaml',
        'weights': '/u/yacheng/projects/ssl_outthere/encoder_image/astrodino/model/astrodino_f115w_vitl/eval/training_79999/teacher_checkpoint.pth',
        'data_root': '/u/yacheng/projects/ssl_outthere/images/jwst/f115w'
    },
    'f150w': {
        'config': '/u/yacheng/projects/ssl_outthere/encoder_image/astrodino/model/astrodino_f150w_vitb/config.yaml',
        'weights': '/u/yacheng/projects/ssl_outthere/encoder_image/astrodino/model/astrodino_f150w_vitb/eval/training_79999/teacher_checkpoint.pth',
        'data_root': '/u/yacheng/projects/ssl_outthere/images/jwst/f150w'
    }
}

BATCH_SIZE = 64
MAX_SAMPLES = 5000  # Number of samples to use for collapse analysis
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Simple Dataset for Collapse Analysis

In [ ]:
class SimpleJWSTDataset(Dataset):
    """Simple dataset for loading JWST images without label filtering."""
    
    def __init__(self, root: str, crop_size: int = 90, max_samples: int = 5000, seed: int = 42):
        self.crop_size = crop_size
        self.to_rgb = ToRGB()
        self.center_crop = transforms.CenterCrop(crop_size)
        self.rng = np.random.default_rng(seed=seed)
        
        # Load all h5 files
        self._files = []
        self._file_names = []
        h5_files = sorted(f for f in os.listdir(root) if f.endswith('.h5'))
        
        for fname in h5_files:
            fpath = os.path.join(root, fname)
            try:
                f = h5py.File(fpath, 'r')
                if 'image' in f.keys():
                    self._files.append(f)
                    self._file_names.append(fname)
                else:
                    f.close()
            except Exception as e:
                print(f'Error loading {fname}: {e}')
        
        print(f'Loaded {len(self._files)} files')
        
        # Build index of all samples
        self._indices = []  # (file_idx, local_idx)
        for file_idx, f in enumerate(self._files):
            n_samples = f['image'].shape[0]
            for local_idx in range(n_samples):
                self._indices.append((file_idx, local_idx))
        
        print(f'Total samples: {len(self._indices)}')
        
        # Random subsample
        if max_samples > 0 and max_samples < len(self._indices):
            indices = self.rng.choice(len(self._indices), size=max_samples, replace=False)
            self._indices = [self._indices[i] for i in indices]
            print(f'Subsampled to {len(self._indices)} samples')
    
    def __len__(self):
        return len(self._indices)
    
    def __getitem__(self, index):
        file_idx, local_idx = self._indices[index]
        img = self._files[file_idx]['image'][local_idx].astype('float32')
        
        # Convert to 3 channel
        img = np.repeat(img[np.newaxis, :, :], 3, axis=0)
        tensor = torch.from_numpy(img)
        tensor = self.center_crop(tensor)
        tensor = torch.from_numpy(self.to_rgb(tensor.numpy()))
        
        return tensor
    
    def close(self):
        for f in self._files:
            try:
                f.close()
            except:
                pass

## 2. Helper Functions for Collapse Analysis

In [ ]:
def compute_embeddings(model, dataset, device, batch_size=64):
    """Compute embeddings for all samples in dataset."""
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    all_embeddings = []
    model.eval()
    
    with torch.no_grad():
        for batch_imgs in tqdm(dataloader, desc='Computing embeddings'):
            batch_imgs = batch_imgs.to(device)
            emb = model(batch_imgs)
            if isinstance(emb, tuple):
                emb = emb[0]
            if emb.dim() > 2:
                emb = emb.view(emb.size(0), -1)
            all_embeddings.append(emb.cpu().numpy())
    
    return np.concatenate(all_embeddings, axis=0)


def l2_normalize(embeddings):
    """L2 normalize embeddings along the feature dimension."""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / (norms + 1e-8)


def compute_nn_gap(embeddings, k_neighbors=10, n_sample=1000):
    """Compute NN gap ratio for given embeddings."""
    from sklearn.neighbors import NearestNeighbors
    
    n_samples = embeddings.shape[0]
    n_sample = min(n_sample, n_samples)
    idx = np.random.choice(n_samples, n_sample, replace=False)
    emb_sample = embeddings[idx]
    
    nn = NearestNeighbors(n_neighbors=k_neighbors+1, metric='euclidean')
    nn.fit(emb_sample)
    distances, _ = nn.kneighbors(emb_sample)
    
    d1 = distances[:, 1]
    dk = distances[:, k_neighbors]
    nn_gap_ratio = (dk - d1) / (d1 + 1e-8)
    
    return d1, dk, nn_gap_ratio


def generate_random_baseline(n_samples, n_dims, seed=None):
    """Generate random isotropic baseline embeddings (always L2 normalized)."""
    rng = np.random.default_rng(seed=seed)
    z = rng.standard_normal((n_samples, n_dims))
    # Always L2 normalize for consistent comparison with model embeddings
    z = l2_normalize(z)
    return z


def analyze_collapse(embeddings, name='Model', k_neighbors=10, normalize=True):
    """Analyze embeddings for signs of collapse with comprehensive metrics.
    
    Args:
        embeddings: Input embeddings of shape (n_samples, n_dims)
        name: Name for display
        k_neighbors: Number of neighbors for NN analysis
        normalize: If True, L2 normalize embeddings before analysis (recommended)
    """
    from sklearn.neighbors import NearestNeighbors
    
    n_samples, n_dims = embeddings.shape
    print(f'\n{"="*70}')
    print(f'Collapse Analysis for {name}')
    print(f'Embeddings shape: {embeddings.shape}')
    print(f'{"="*70}')
    
    # Check original embedding norms
    original_norms = np.linalg.norm(embeddings, axis=1)
    was_normalized = np.allclose(original_norms, 1.0, atol=0.1)
    print(f'Original L2 normalized: {was_normalized} (mean norm: {original_norms.mean():.3f})')
    
    # L2 normalize if requested
    if normalize and not was_normalized:
        print(f'Applying L2 normalization for analysis...')
        embeddings = l2_normalize(embeddings)
    elif normalize and was_normalized:
        print(f'Embeddings already L2 normalized.')
    else:
        print(f'Skipping L2 normalization (normalize=False)')
    
    # Sample for efficiency
    n_sample = min(1000, n_samples)
    idx = np.random.choice(n_samples, n_sample, replace=False)
    emb_sample = embeddings[idx]
    
    # ========== 1. Pairwise Cosine Similarity ==========
    print('\n[1] Pairwise Cosine Similarity')
    cos_sim = cosine_similarity(emb_sample)
    triu_idx = np.triu_indices(n_sample, k=1)
    cos_sim_vals = cos_sim[triu_idx]
    
    cos_mean = cos_sim_vals.mean()
    cos_std = cos_sim_vals.std()
    frac_09 = (cos_sim_vals > 0.9).mean()
    frac_095 = (cos_sim_vals > 0.95).mean()
    
    print(f'  Mean: {cos_mean:.4f}')
    print(f'  Std:  {cos_std:.4f}')
    print(f'  Min:  {cos_sim_vals.min():.4f}, Max: {cos_sim_vals.max():.4f}')
    print(f'  Fraction > 0.9:  {frac_09*100:.2f}%')
    print(f'  Fraction > 0.95: {frac_095*100:.2f}%')
    
    # ========== 2. Per-dimension Variance ==========
    print('\n[2] Per-dimension Variance')
    emb_centered = embeddings - embeddings.mean(axis=0)
    dim_variance = np.var(emb_centered, axis=0)
    
    # For L2 normalized embeddings, expected variance per dim ~ 1/n_dims
    expected_var = 1.0 / n_dims if normalize else None
    dead_threshold = expected_var * 0.01 if normalize else 1e-5  # 1% of expected
    low_threshold = expected_var * 0.1 if normalize else 1e-3   # 10% of expected
    
    dead_dims = (dim_variance < dead_threshold).sum()
    low_dims = (dim_variance < low_threshold).sum()
    
    print(f'  Mean variance: {dim_variance.mean():.6f}')
    print(f'  Std of variance: {dim_variance.std():.6f}')
    if normalize:
        print(f'  Expected variance (isotropic): {expected_var:.6f}')
    print(f'  Dead dims (var < {dead_threshold:.2e}): {dead_dims} / {n_dims}')
    print(f'  Low dims (var < {low_threshold:.2e}): {low_dims} / {n_dims}')
    
    # ========== 3. Effective Dimensionality ==========
    print('\n[3] Effective Dimensionality')
    cov_matrix = np.cov(embeddings.T)
    eigenvalues = np.linalg.eigvalsh(cov_matrix)
    eigenvalues = np.sort(eigenvalues)[::-1]
    eigenvalues = np.maximum(eigenvalues, 1e-12)  # Numerical stability
    
    total_var = eigenvalues.sum()
    normalized_eig = eigenvalues / total_var
    
    # Method 1: Participation Ratio (PR)
    participation_ratio = 1.0 / np.sum(normalized_eig ** 2)
    
    # Method 2: Effective Rank via Entropy (ER)
    entropy = -np.sum(normalized_eig * np.log(normalized_eig + 1e-12))
    effective_rank = np.exp(entropy)
    
    # Variance explained
    cumsum = np.cumsum(eigenvalues) / total_var
    dims_90 = np.searchsorted(cumsum, 0.90) + 1
    dims_95 = np.searchsorted(cumsum, 0.95) + 1
    dims_99 = np.searchsorted(cumsum, 0.99) + 1
    
    print(f'  Participation Ratio (PR): {participation_ratio:.2f} / {n_dims}')
    print(f'  Effective Rank (Entropy): {effective_rank:.2f} / {n_dims}')
    print(f'  Dims for 90% variance: {dims_90}')
    print(f'  Dims for 95% variance: {dims_95}')
    print(f'  Dims for 99% variance: {dims_99}')
    print(f'  Top eigenvalue ratio: {eigenvalues[0]/total_var*100:.2f}%')
    
    # ========== 4. Nearest Neighbor Analysis ==========
    print(f'\n[4] Nearest Neighbor Analysis (k={k_neighbors})')
    d1, dk, nn_gap_ratio = compute_nn_gap(embeddings, k_neighbors, n_sample)
    
    print(f'  d_1 (1st NN): mean={d1.mean():.4f}, std={d1.std():.4f}')
    print(f'  d_{k_neighbors} ({k_neighbors}th NN): mean={dk.mean():.4f}, std={dk.std():.4f}')
    print(f'  NN Gap Ratio (d_k-d_1)/d_1:')
    print(f'    Mean: {nn_gap_ratio.mean():.4f}')
    print(f'    Std:  {nn_gap_ratio.std():.4f}')
    print(f'  Distance Variance Var(d_1): {d1.var():.4f}')
    
    # ========== 5. Random Baseline Comparison ==========
    print(f'\n[5] Random Baseline Comparison (L2 normalized)')
    # Random baseline is always L2 normalized for fair comparison
    random_emb = generate_random_baseline(n_samples, n_dims, seed=42)
    d1_rand, dk_rand, nn_gap_rand = compute_nn_gap(random_emb, k_neighbors, n_sample)
    
    # Per-dimension variance for random baseline
    random_centered = random_emb - random_emb.mean(axis=0)
    dim_variance_rand = np.var(random_centered, axis=0)
    
    # Cosine similarity for random baseline
    random_sample = random_emb[np.random.choice(n_samples, n_sample, replace=False)]
    cos_sim_rand = cosine_similarity(random_sample)
    cos_sim_rand_vals = cos_sim_rand[np.triu_indices(n_sample, k=1)]
    
    print(f'  [Random Baseline]')
    print(f'    Cosine Sim: mean={cos_sim_rand_vals.mean():.4f}, std={cos_sim_rand_vals.std():.4f}')
    print(f'    d_1: mean={d1_rand.mean():.4f}, std={d1_rand.std():.4f}')
    print(f'    NN Gap Ratio: mean={nn_gap_rand.mean():.4f}, std={nn_gap_rand.std():.4f}')
    print(f'    Dim Variance: mean={dim_variance_rand.mean():.6f}')
    
    print(f'\n  [Model vs Random]')
    gap_ratio_diff = nn_gap_ratio.mean() - nn_gap_rand.mean()
    cos_sim_diff = cos_mean - cos_sim_rand_vals.mean()
    var_ratio = dim_variance.mean() / dim_variance_rand.mean()
    print(f'    NN Gap Ratio: model={nn_gap_ratio.mean():.4f}, random={nn_gap_rand.mean():.4f}, diff={gap_ratio_diff:+.4f}')
    print(f'    Cosine Sim:   model={cos_mean:.4f}, random={cos_sim_rand_vals.mean():.4f}, diff={cos_sim_diff:+.4f}')
    print(f'    Dim Variance: model={dim_variance.mean():.6f}, random={dim_variance_rand.mean():.6f}, ratio={var_ratio:.2f}x')
    
    if nn_gap_ratio.mean() > nn_gap_rand.mean():
        print(f'    → Model shows MORE local structure than random baseline')
    else:
        print(f'    → Model shows LESS local structure than random baseline')
    
    return {
        'cos_sim': cos_sim_vals,
        'cos_mean': cos_mean,
        'cos_std': cos_std,
        'dim_variance': dim_variance,
        'eigenvalues': eigenvalues,
        'cumsum_var': cumsum,
        'participation_ratio': participation_ratio,
        'effective_rank': effective_rank,
        'd1': d1,
        'dk': dk,
        'nn_gap_ratio': nn_gap_ratio,
        'k_neighbors': k_neighbors,
        'dims_90': dims_90,
        'dims_95': dims_95,
        'dims_99': dims_99,
        # Baseline results
        'd1_rand': d1_rand,
        'dk_rand': dk_rand,
        'nn_gap_rand': nn_gap_rand,
        'cos_sim_rand': cos_sim_rand_vals,
        'dim_variance_rand': dim_variance_rand,
        # Metadata
        'was_normalized': was_normalized,
        'normalize_applied': normalize,
    }

In [ ]:
def plot_collapse_analysis(results, name='Model'):
    """Plot collapse analysis results WITHOUT baseline comparison.
    
    Layout (2 rows x 3 cols):
    - Row 1: Pairwise Cosine Similarity, NN Distance, NN Gap Ratio
    - Row 2: Per-dimension Variance, Eigenvalue Spectrum, Effective Dimensionality
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Collapse Analysis: {name}', fontsize=14, fontweight='bold')
    
    # ========== Row 0: Similarity & Distance Metrics ==========
    
    # Row 0, Col 0: Cosine Similarity Histogram
    ax = axes[0, 0]
    ax.hist(results['cos_sim'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(x=0.9, color='r', linestyle='--', linewidth=2, label='0.9 (danger)')
    ax.axvline(x=results['cos_mean'], color='lime', linestyle='-', linewidth=2, 
               label=f'Mean: {results["cos_mean"]:.3f}')
    ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Cosine Similarity', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'Pairwise Cosine Similarity\n(std={results["cos_std"]:.3f})', fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Row 0, Col 1: NN Distance Comparison (d1 vs dk)
    ax = axes[0, 1]
    k = results['k_neighbors']
    ax.hist(results['d1'], bins=40, alpha=0.6, label=f'd_1 (1st NN)', color='blue', edgecolor='black')
    ax.hist(results['dk'], bins=40, alpha=0.6, label=f'd_{k} ({k}th NN)', color='red', edgecolor='black')
    ax.axvline(x=results['d1'].mean(), color='blue', linestyle='-', linewidth=2)
    ax.axvline(x=results['dk'].mean(), color='red', linestyle='-', linewidth=2)
    ax.set_xlabel('L2 Distance', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'NN Distance Distribution\n(gap indicates local structure)', fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Row 0, Col 2: NN Gap Ratio Distribution
    ax = axes[0, 2]
    gap_ratio = results['nn_gap_ratio']
    ax.hist(gap_ratio, bins=40, edgecolor='black', alpha=0.7, color='forestgreen')
    ax.axvline(x=gap_ratio.mean(), color='lime', linestyle='-', linewidth=2, 
               label=f'Mean: {gap_ratio.mean():.3f}')
    ax.set_xlabel(f'Gap Ratio: (d_{k} - d_1) / d_1', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'NN Gap Ratio (k={k})', fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # ========== Row 1: Variance & Dimensionality Metrics ==========
    
    # Row 1, Col 0: Per-dimension Variance (sorted)
    ax = axes[1, 0]
    sorted_var = np.sort(results['dim_variance'])[::-1]
    ax.semilogy(sorted_var, color='darkorange', linewidth=1.5)
    ax.axhline(y=1e-5, color='r', linestyle='--', linewidth=2, label='1e-5 (dead)')
    ax.axhline(y=1e-3, color='orange', linestyle='--', linewidth=1.5, label='1e-3')
    ax.fill_between(range(len(sorted_var)), sorted_var, alpha=0.3, color='orange')
    ax.set_xlabel('Dimension (sorted by variance)', fontsize=11)
    ax.set_ylabel('Variance (log scale)', fontsize=11)
    ax.set_title('Per-dimension Variance', fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Row 1, Col 1: Eigenvalue Spectrum
    ax = axes[1, 1]
    n_show = min(200, len(results['eigenvalues']))
    ax.semilogy(results['eigenvalues'][:n_show], color='purple', linewidth=2)
    ax.fill_between(range(n_show), results['eigenvalues'][:n_show], alpha=0.3, color='purple')
    ax.set_xlabel('Eigenvalue Index', fontsize=11)
    ax.set_ylabel('Eigenvalue (log scale)', fontsize=11)
    ax.set_title(f'Covariance Eigenvalue Spectrum\n(Top 1: {results["eigenvalues"][0]/results["eigenvalues"].sum()*100:.1f}%)', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Row 1, Col 2: Cumulative Variance Explained with Effective Rank
    ax = axes[1, 2]
    n_show = min(300, len(results['cumsum_var']))
    ax.plot(results['cumsum_var'][:n_show], linewidth=2, color='teal')
    ax.axhline(y=0.90, color='r', linestyle='--', alpha=0.7, label='90%')
    ax.axhline(y=0.95, color='orange', linestyle='--', alpha=0.7, label='95%')
    ax.axhline(y=0.99, color='g', linestyle='--', alpha=0.7, label='99%')
    ax.axvline(x=results['effective_rank'], color='blue', linestyle='-', linewidth=2, 
               label=f'Eff. Rank: {results["effective_rank"]:.0f}')
    ax.axvline(x=results['participation_ratio'], color='cyan', linestyle='--', linewidth=1.5,
               label=f'Part. Ratio: {results["participation_ratio"]:.0f}')
    ax.set_xlabel('Number of Principal Components', fontsize=11)
    ax.set_ylabel('Cumulative Variance Explained', fontsize=11)
    ax.set_title(f'Effective Dimensionality\n(90%@{results["dims_90"]}, 95%@{results["dims_95"]})', fontsize=12)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, n_show)
    
    plt.tight_layout()
    plt.show()
    
    return fig


def plot_collapse_analysis_with_baseline(results, name='Model'):
    """Plot collapse analysis results WITH baseline comparison.
    
    Layout (2 rows x 3 cols):
    - Row 1: Pairwise Cosine Similarity, NN Distance, NN Gap Ratio (all with baseline)
    - Row 2: Per-dimension Variance, Eigenvalue Spectrum, Effective Dimensionality
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Collapse Analysis: {name} (with Random Baseline)', fontsize=14, fontweight='bold')
    
    # ========== Row 0: Similarity & Distance Metrics (with baseline) ==========
    
    # Row 0, Col 0: Cosine Similarity Histogram (with baseline)
    ax = axes[0, 0]
    ax.hist(results['cos_sim'], bins=50, alpha=0.7, color='steelblue', 
            edgecolor='black', label=f'Model (μ={results["cos_mean"]:.3f})', density=True)
    ax.hist(results['cos_sim_rand'], bins=50, alpha=0.5, color='gray', 
            edgecolor='black', label=f'Random (μ={results["cos_sim_rand"].mean():.3f})', density=True)
    ax.axvline(x=0.9, color='r', linestyle='--', linewidth=2, label='0.9 (danger)')
    ax.set_xlabel('Cosine Similarity', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'Pairwise Cosine Similarity\n(Model vs Random Baseline)', fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Row 0, Col 1: NN Distance Comparison (Model vs Baseline)
    ax = axes[0, 1]
    ax.hist(results['d1'], bins=40, alpha=0.7, label=f'Model d_1 (μ={results["d1"].mean():.3f})', 
            color='blue', edgecolor='black', density=True)
    ax.hist(results['d1_rand'], bins=40, alpha=0.5, label=f'Random d_1 (μ={results["d1_rand"].mean():.3f})', 
            color='gray', edgecolor='black', density=True)
    ax.set_xlabel('L2 Distance to 1st NN', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'1st NN Distance Distribution\n(Model vs Random Baseline)', fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Row 0, Col 2: NN Gap Ratio Comparison (Model vs Baseline)
    ax = axes[0, 2]
    k = results['k_neighbors']
    ax.hist(results['nn_gap_ratio'], bins=40, alpha=0.7, color='forestgreen', 
            edgecolor='black', label=f'Model (μ={results["nn_gap_ratio"].mean():.3f})', density=True)
    ax.hist(results['nn_gap_rand'], bins=40, alpha=0.5, color='gray', 
            edgecolor='black', label=f'Random (μ={results["nn_gap_rand"].mean():.3f})', density=True)
    ax.axvline(x=results['nn_gap_ratio'].mean(), color='lime', linestyle='-', linewidth=2)
    ax.axvline(x=results['nn_gap_rand'].mean(), color='black', linestyle='--', linewidth=2)
    ax.set_xlabel(f'Gap Ratio: (d_{k} - d_1) / d_1', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'NN Gap Ratio (k={k})\n(Model vs Random Baseline)', fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # ========== Row 1: Variance & Dimensionality Metrics ==========
    
    # Row 1, Col 0: Per-dimension Variance (sorted, with baseline)
    ax = axes[1, 0]
    sorted_var = np.sort(results['dim_variance'])[::-1]
    sorted_var_rand = np.sort(results['dim_variance_rand'])[::-1]
    ax.semilogy(sorted_var, color='darkorange', linewidth=1.5, label=f'Model (μ={results["dim_variance"].mean():.2e})')
    ax.semilogy(sorted_var_rand, color='gray', linewidth=1.5, alpha=0.7, label=f'Random (μ={results["dim_variance_rand"].mean():.2e})')
    ax.axhline(y=1e-5, color='r', linestyle='--', linewidth=2, label='1e-5 (dead)')
    ax.fill_between(range(len(sorted_var)), sorted_var, alpha=0.3, color='orange')
    ax.set_xlabel('Dimension (sorted by variance)', fontsize=11)
    ax.set_ylabel('Variance (log scale)', fontsize=11)
    ax.set_title('Per-dimension Variance\n(Model vs Random Baseline)', fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Row 1, Col 1: Eigenvalue Spectrum
    ax = axes[1, 1]
    n_show = min(200, len(results['eigenvalues']))
    ax.semilogy(results['eigenvalues'][:n_show], color='purple', linewidth=2)
    ax.fill_between(range(n_show), results['eigenvalues'][:n_show], alpha=0.3, color='purple')
    ax.set_xlabel('Eigenvalue Index', fontsize=11)
    ax.set_ylabel('Eigenvalue (log scale)', fontsize=11)
    ax.set_title(f'Covariance Eigenvalue Spectrum\n(Top 1: {results["eigenvalues"][0]/results["eigenvalues"].sum()*100:.1f}%)', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Row 1, Col 2: Cumulative Variance Explained with Effective Rank
    ax = axes[1, 2]
    n_show = min(300, len(results['cumsum_var']))
    ax.plot(results['cumsum_var'][:n_show], linewidth=2, color='teal')
    ax.axhline(y=0.90, color='r', linestyle='--', alpha=0.7, label='90%')
    ax.axhline(y=0.95, color='orange', linestyle='--', alpha=0.7, label='95%')
    ax.axhline(y=0.99, color='g', linestyle='--', alpha=0.7, label='99%')
    ax.axvline(x=results['effective_rank'], color='blue', linestyle='-', linewidth=2, 
               label=f'Eff. Rank: {results["effective_rank"]:.0f}')
    ax.axvline(x=results['participation_ratio'], color='cyan', linestyle='--', linewidth=1.5,
               label=f'Part. Ratio: {results["participation_ratio"]:.0f}')
    ax.set_xlabel('Number of Principal Components', fontsize=11)
    ax.set_ylabel('Cumulative Variance Explained', fontsize=11)
    ax.set_title(f'Effective Dimensionality\n(90%@{results["dims_90"]}, 95%@{results["dims_95"]})', fontsize=12)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, n_show)
    
    plt.tight_layout()
    plt.show()
    
    return fig

## 3. Analyze f115w Model

In [ ]:
# Load f115w model
print('Loading f115w model...')
cfg_f115w = OmegaConf.load(MODELS['f115w']['config'])
model_f115w = build_model_for_eval(cfg_f115w, pretrained_weights=MODELS['f115w']['weights'])
model_f115w = model_f115w.to(DEVICE)
model_f115w.eval()
print(f'f115w model loaded, crop_size={cfg_f115w.crops.global_crops_size}')

In [ ]:
# Create dataset for f115w
dataset_f115w = SimpleJWSTDataset(
    MODELS['f115w']['data_root'],
    crop_size=cfg_f115w.crops.global_crops_size,
    max_samples=MAX_SAMPLES
)
print(f'f115w dataset size: {len(dataset_f115w)}')

In [ ]:
# Compute embeddings for f115w
embeddings_f115w = compute_embeddings(model_f115w, dataset_f115w, DEVICE, BATCH_SIZE)
print(f'f115w embeddings shape: {embeddings_f115w.shape}')

In [ ]:
# Analyze collapse for f115w
results_f115w = analyze_collapse(embeddings_f115w, name='AstroDINO f115w')

In [ ]:
# Plot collapse analysis for f115w
fig_f115w = plot_collapse_analysis(results_f115w, name='AstroDINO f115w')
fig_f115w_with_base_line = plot_collapse_analysis_with_baseline(results_f115w, name='AstroDINO f115w')

In [ ]:
# Clean up f115w
dataset_f115w.close()
del model_f115w
torch.cuda.empty_cache()
print('f115w resources released')

## 4. Analyze f150w Model

In [ ]:
# Load f150w model
print('Loading f150w model...')
cfg_f150w = OmegaConf.load(MODELS['f150w']['config'])
model_f150w = build_model_for_eval(cfg_f150w, pretrained_weights=MODELS['f150w']['weights'])
model_f150w = model_f150w.to(DEVICE)
model_f150w.eval()
print(f'f150w model loaded, crop_size={cfg_f150w.crops.global_crops_size}')

In [ ]:
# Create dataset for f150w
dataset_f150w = SimpleJWSTDataset(
    MODELS['f150w']['data_root'],
    crop_size=cfg_f150w.crops.global_crops_size,
    max_samples=MAX_SAMPLES
)
print(f'f150w dataset size: {len(dataset_f150w)}')

In [ ]:
# Compute embeddings for f150w
embeddings_f150w = compute_embeddings(model_f150w, dataset_f150w, DEVICE, BATCH_SIZE)
print(f'f150w embeddings shape: {embeddings_f150w.shape}')

In [ ]:
# Analyze collapse for f150w
results_f150w = analyze_collapse(embeddings_f150w, name='AstroDINO f150w')

In [ ]:
# Plot collapse analysis for f150w
fig_f150w = plot_collapse_analysis(results_f150w, name='AstroDINO f150w')

In [ ]:
# Clean up f150w
dataset_f150w.close()
print('f150w resources released')

## 5. Comparison Summary

In [ ]:
# Compare both models side by side
print('\n' + '='*70)
print('COLLAPSE ANALYSIS SUMMARY')
print('='*70)

print(f"\n{'Metric':<40} {'f115w':>12} {'f150w':>12}")
print('-'*70)

print(f"{'Mean Cosine Similarity':<40} {results_f115w['cos_sim'].mean():>12.4f} {results_f150w['cos_sim'].mean():>12.4f}")
print(f"{'Std Cosine Similarity':<40} {results_f115w['cos_sim'].std():>12.4f} {results_f150w['cos_sim'].std():>12.4f}")
print(f"{'Fraction cos_sim > 0.9':<40} {(results_f115w['cos_sim'] > 0.9).mean()*100:>11.2f}% {(results_f150w['cos_sim'] > 0.9).mean()*100:>11.2f}%")
print(f"{'Effective Dimensionality':<40} {results_f115w['participation_ratio']:>12.1f} {results_f150w['participation_ratio']:>12.1f}")
print(f"{'Mean NN Distance':<40} {results_f115w['d1'].mean():>12.4f} {results_f150w['d1'].mean():>12.4f}")
print(f"{'Std NN Distance':<40} {results_f115w['d1'].std():>12.4f} {results_f150w['d1'].std():>12.4f}")

print('\n' + '='*70)
print('COLLAPSE DIAGNOSIS')
print('='*70)

for name, results in [('f115w', results_f115w), ('f150w', results_f150w)]:
    cos_mean = results['cos_sim'].mean()
    cos_frac_09 = (results['cos_sim'] > 0.9).mean()
    eff_dim = results['participation_ratio']
    
    print(f'\n{name}:')
    if cos_mean > 0.9 or cos_frac_09 > 0.5:
        print(f'  ⚠️  HIGH cosine similarity (mean={cos_mean:.3f}, {cos_frac_09*100:.1f}% > 0.9) - POSSIBLE COLLAPSE')
    else:
        print(f'  ✓ Cosine similarity looks healthy (mean={cos_mean:.3f})')
    
    if eff_dim < 10:
        print(f'  ⚠️  LOW effective dimensionality ({eff_dim:.1f}) - POSSIBLE COLLAPSE')
    elif eff_dim < 50:
        print(f'  ⚠ Moderate effective dimensionality ({eff_dim:.1f})')
    else:
        print(f'  ✓ Effective dimensionality looks healthy ({eff_dim:.1f})')

In [ ]:
# Side-by-side comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Collapse Comparison: f115w vs f150w', fontsize=14, fontweight='bold')

# Cosine similarity comparison
ax = axes[0, 0]
ax.hist(results_f115w['cos_sim'], bins=50, alpha=0.6, label='f115w', density=True)
ax.hist(results_f150w['cos_sim'], bins=50, alpha=0.6, label='f150w', density=True)
ax.axvline(x=0.9, color='r', linestyle='--', label='0.9 threshold')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Density')
ax.set_title('Pairwise Cosine Similarity')
ax.legend()
ax.grid(True, alpha=0.3)

# Eigenvalue spectrum comparison
ax = axes[0, 1]
ax.semilogy(results_f115w['eigenvalues'][:200], label='f115w', linewidth=2)
ax.semilogy(results_f150w['eigenvalues'][:200], label='f150w', linewidth=2)
ax.set_xlabel('Eigenvalue Index')
ax.set_ylabel('Eigenvalue (log scale)')
ax.set_title('Covariance Eigenvalue Spectrum')
ax.legend()
ax.grid(True, alpha=0.3)

# Cumulative variance comparison
ax = axes[1, 0]
ax.plot(results_f115w['cumsum_var'][:200], label='f115w', linewidth=2)
ax.plot(results_f150w['cumsum_var'][:200], label='f150w', linewidth=2)
ax.axhline(y=0.95, color='r', linestyle='--', alpha=0.7)
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Variance Explained')
ax.set_title('Cumulative Variance Explained')
ax.legend()
ax.grid(True, alpha=0.3)

# NN distance comparison
ax = axes[1, 1]
ax.hist(results_f115w['d1'], bins=50, alpha=0.6, label='f115w', density=True)
ax.hist(results_f150w['d1'], bins=50, alpha=0.6, label='f150w', density=True)
ax.set_xlabel('L2 Distance to Nearest Neighbor')
ax.set_ylabel('Density')
ax.set_title('Nearest Neighbor Distance')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()